In [ ]:
# Goal: Generate timeseries data from realistic load profiles (df_load_buses.parquet)

# Steps:
# 1. Pick n load buses randomly from the realistic load profiles (df_load_buses.parquet)
    # - load_buses.parquet load profiles for 2000 buses. 
    # - The amount of load buses needed is derived from the selected IEEE case (e.g. IEEE case 14 ⇒  11 load buses) 
# 2. Normalise every load bus profile individually 
    # - normalise every bus load profile by dividing every load by its buses max_load
# 3. Mulitply the normalised bus load profiles with the Active Power (P) and Reactive Power (Q), derived from the selected IEEE case
    # - for case 14: The non-load buses Bus 0, bus 6 and bus 7 (in zero indexing) will have P and Q zero
# 4. Formatting: Use the normalised load buses timesieries to create the precomputed_profile.csv
    # - Formatting defined in load_pertubation.py → *class* *PrecomputedProfile(LoadScenarioGeneratorBase)*
# 5 Generate Output via gridfm-datakit with config.yaml:
    # load: generator: precomputed_profile
    # Added argument: scenario_file: "/path/to/load-scenarios/load-scenarios-precomputed-temp.csv" # precomputed scenarios (cols: load_scenario, load, p_mw, q_mvar)
    # All pertubations set to None

In [ ]:
import pandas as pd
import numpy as np
from pypower.api import case14

def get_ieee14_base_loads():
    """
    Returns a DataFrame with base P (MW) and Q (MVar) for all 14 buses.
    Maps Matpower 1-based IDs to 0-based indices.
    """
    mpc = case14()
    # PD is col 2, QD is col 3
    bus_ids = mpc['bus'][:, 0].astype(int)
    P_base = mpc['bus'][:, 2]
    Q_base = mpc['bus'][:, 3]
    base_df = pd.DataFrame({
        'bus_idx': np.arange(len(bus_ids)), # 0..13
        'P_base': P_base,
        'Q_base': Q_base
    })
    return base_df

ieee_base = get_ieee14_base_loads()


In [ ]:
print(ieee_base.head())

    bus_idx  P_base  Q_base
0         0     0.0     0.0
1         1    21.7    12.7
2         2    94.2    19.0
3         3    47.8    -3.9
4         4     7.6     1.6
5         5    11.2     7.5
6         6     0.0     0.0
7         7     0.0     0.0
8         8    29.5    16.6
9         9     9.0     5.8
10       10     3.5     1.8
11       11     6.1     1.6
12       12    13.5     5.8
13       13    14.9     5.0


In [ ]:
# config
df = pd.read_parquet("df_load_buses.parquet")
N_BUSES = 11 # IEEE-14 has 11 load buses
OUTPUT_FILE = "precomputed_load_profiles.csv"

# sample  N_BUSES
wide_df = df.pivot(index='timestamp', columns='bus_id', values='load').sort_index()
sampled_df = wide_df.sample(n=N_BUSES, axis=1, random_state=42)

# normalise
norm_profiles = (sampled_df / sampled_df.max()).values 
n_scenarios = len(norm_profiles)

print(f"Time steps: {n_scenarios}, \n normalised Shape: {norm_profiles.shape}")

Ready. Time steps: 8760, Normalized Shape: (8760, 11)


In [ ]:
sampled_df.head()

Index([639, 247, 1612, 316, 765, 804, 1472, 1290, 225, 1536, 1799], dtype='int64', name='bus_id')

In [ ]:
# multiply realistic load profiles with IEEE case values (P,Q) -> create output file
ieee = ieee_base.sort_values('bus_idx')
load_mask = (ieee['P_base'] != 0).values # Boolean mask for the load buses

P_mat = np.zeros((n_scenarios, 14))
Q_mat = np.zeros((n_scenarios, 14))
P_mat[:, load_mask] = norm_profiles * ieee.loc[load_mask, 'P_base'].values
Q_mat[:, load_mask] = norm_profiles * ieee.loc[load_mask, 'Q_base'].values

out_df = pd.DataFrame({
    'load_scenario': np.repeat(np.arange(n_scenarios), 14),
    'load':          np.tile(ieee['bus_idx'].values, n_scenarios),
    'p_mw':          P_mat.flatten(), # Flatten -> (Time 0 [Bus0..n], Time 1 [Bus0..n]...)
    'q_mvar':        Q_mat.flatten()
})

out_df.to_csv(OUTPUT_FILE, index=False)
print(f"Output_df {len(
    
)} rows.")
print(f"Expected rows: {n_scenarios * 14}")

Output_df 122640 rows.
Expected rows: 122640
